![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 05: Knowledge Agents and Stateful Workflows)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 5C: LangGraph Stateful Workflows

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Optional LangGraph package implementation.</td></tr>
<tr><td align="left">Main output</td><td>Represent agent workflows as state transitions and branches.</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m05c-overview)
2. [Conceptual Background](#m05c-background)
3. [Setup](#m05c-setup)
4. [Approved Local Data](#m05c-data)
5. [Mandatory Local Workflow](#m05c-workflow)
6. [Inspection and Interpretation](#m05c-inspection)
7. [Optional Real Model or Package Section](#m05c-optional)
8. [Testing and Analysis](#m05c-testing)
9. [Student Tasks](#m05c-tasks)
10. [Submission and Reflection](#m05c-submission)

---

<a id="m05c-overview"></a>

### 1. Overview and Learning Goals

This session is **M05C: LangGraph Stateful Workflows**. In M05A and M05B you built pipelines: straight lines where each step feeds the next. Real agent workflows are rarely straight lines. They branch — a request may be refused, may lack evidence, or may complete — and every step needs to know what earlier steps decided. This session teaches the vocabulary and structure for that: **graphs of nodes and edges operating on shared state**.

The central theme is:

```text
Represent agent workflows as state transitions and branches.
```

A useful analogy is a shared whiteboard. Imagine a team solving a problem where each specialist walks up to the whiteboard, reads what is already written, adds their contribution, and steps back. The whiteboard is the **state**. Each specialist is a **node**. The rule for deciding who goes next — "if the request was rejected, the refusal writer goes next; otherwise the researcher does" — is an **edge**. Nobody keeps private notes: everything a later node needs must be on the whiteboard.

The workflow you will build has exactly this shape:

```text
                     [ validate ]
                      /        \
             allowed /          \ not allowed
                    v            v
              [ select ]     (refused result)
               /      \
    items found        nothing relevant
             v              v
      [ build result ]   (insufficient-context result)
             v
      (completed result)
```

The session is intentionally designed with a mandatory local workflow first: the graph above is implemented in plain Python, so every node and every edge is a line of code you can read. The optional LangGraph package expresses the same design with `StateGraph`, but the design is the learning objective — not the package.

The concepts used in this session are:

```text
1. state
2. node
3. edge
4. branch
5. validation error
6. refusal
```

By the end of this session, you should be able to describe the workflow in your own words, run the mandatory local implementation, inspect intermediate outputs, add a small extension, test normal and failure cases, and explain how the design would change if a real model or external package were added.


<a id="m05c-background"></a>

### 2. Conceptual Background

The important point in this practical is not just to make a notebook run. The important point is to understand the design discipline behind an agentic AI workflow.

A weak workflow often does this:

```text
User request --> One large prompt --> Model output
```

This is simple, but it hides too many decisions. There is no place to ask: was the input valid? Should this request have been refused? Was there enough context? Everything happens inside one opaque call, so nothing can be tested separately.

A stronger workflow makes each decision a visible step, connected by explicit transitions:

```text
User request
     |
     v
[ validate node ] --- state.allowed = False ---> [ refuse node ] --> END
     |
     | state.allowed = True
     v
[ select node ] ---- state.items = [] ----> [ insufficient node ] --> END
     |
     | state.items is non-empty
     v
[ build node ]
     |
     v
    END
```

Read the labels on the arrows carefully: every transition is decided by looking at the **state** — the shared record that flows through the graph. Nothing is decided by hidden variables. That is what makes the workflow debuggable: to understand any outcome, you replay the state through the edges.

For **LangGraph Stateful Workflows**, these concepts matter:

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>What it means here</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">state</td><td>The shared record every step reads and updates — in this notebook, the dictionaries carrying <code>request</code>, <code>status</code>, <code>selected_items</code> and <code>limitations</code>. The whiteboard.</td></tr>
<tr><td align="left">node</td><td>One unit of work that reads state and writes an update: <code>validate_request</code>, <code>select_relevant_items</code>, <code>build_structured_result</code>.</td></tr>
<tr><td align="left">edge</td><td>The decision about which node runs next — implemented as the <code>if</code> statements inside <code>run_local_workflow</code>.</td></tr>
<tr><td align="left">branch</td><td>An alternative path through the graph. This workflow has three terminal branches: <code>completed</code>, <code>insufficient_context</code> and <code>refused</code>.</td></tr>
<tr><td align="left">validation error</td><td>Malformed input (<code>ok=False</code>) stops the graph before any real work — a fourth outcome distinct from the three branches.</td></tr>
<tr><td align="left">refusal</td><td>A deliberate terminal branch, not a crash: the graph completes successfully by declining, with a stated reason.</td></tr>
</tbody>
</table>

</div>

The mandatory workflow uses plain Python rather than the LangGraph package because a local simulation makes the control structure visible: every edge is an `if` you can point at. Once you can draw the graph of your own workflow, learning `StateGraph` is mostly learning new spelling for ideas you already have.


<a id="m05c-setup"></a>

### 3. Setup

The mandatory part uses standard Python only, so it runs identically in Colab and local Jupyter, with no packages to install and no API key to manage. The optional section later mentions the LangGraph package, but it is not required.

The safety boundary for this session is:

```text
1. Use only approved public-style or synthetic teaching data.
2. Do not use private documents, credentials, emails, student records or hidden instructor materials.
3. Do not perform real external side effects.
4. Show limitations when the local workflow does not have enough information.
5. Keep output inspectable and testable.
```

The boundary matters more, not less, once workflows branch: a graph with a refusal branch that nobody tests is a graph without a refusal branch. The testing section will exercise every branch explicitly.

Run the setup cell below; you should see `Setup complete.` and nothing else.


In [ ]:
# Standard library only: the mandatory graph is plain Python, so every
# node and edge stays readable and the whole class runs the same code.

import json  # pretty-printing items and structured state
import re    # tokenising text for lexical matching
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("Setup complete.")


<a id="m05c-data"></a>

### 4. Approved Local Data

The local data below is synthetic teaching data for this practical. It is not private data. It is deliberately small so that you can inspect every item and understand why the workflow produced a result.

The three items describe the graph concepts themselves — state, nodes and edges — so the workflow you build will be answering questions about its own building blocks. Each item has:

```text
item_id:    stable identifier, cited as the evidence for a result
title:      short title shown with selection results
content:    the approved teaching content results are built from
tags:       labels that give the lexical matcher extra vocabulary
risk_level: low / medium / high -- how carefully a result built on this
            item should be reviewed before being reused
```

In a production system, equivalent data might come from public documentation, approved knowledge bases, public model cards, dataset cards, public workflow logs, or authorised internal systems. This practical does not use those live sources.

Run the next cell and confirm it reports three items and prints the first one in full.


In [ ]:
# Three small approved items about the graph concepts of this session.
# Small data is a feature: with three items you can predict which one a
# request should select, then check whether the graph agrees with you.

LOCAL_ITEMS = [
    {
        "item_id": "M05C-001",
        "title": "State Basics",
        "content": "This item explains state in the context of LangGraph Stateful Workflows. It is approved synthetic teaching content.",
        "tags": ["state", "basics", "approved"],
        "risk_level": "low"
    },
    {
        "item_id": "M05C-002",
        "title": "Node Practice",
        "content": "This item describes how node can be handled through validation, inspection and structured output.",
        "tags": ["node", "practice", "validation"],
        "risk_level": "low"
    },
    {
        "item_id": "M05C-003",
        "title": "Edge Safety",
        "content": "This item highlights the safety boundary for edge and explains why unsupported claims or external side effects should be avoided.",
        "tags": ["edge", "safety", "boundary"],
        "risk_level": "medium"
    },
]

print("Number of local items:", len(LOCAL_ITEMS))
print(json.dumps(LOCAL_ITEMS[0], indent=2))

The local items play the same role as a small approved knowledge base. The purpose is not to cover every real-world case. The purpose is to make the workflow observable: you can predict which item a request should select and which branch the graph should take, run it, and check. That predict-then-check habit scales directly to larger LangGraph applications, where the graph is bigger but the debugging method is identical.


<a id="m05c-workflow"></a>

### 5. Mandatory Local Workflow

The workflow has four functions. Three of them are the graph's **nodes** — each reads the current state and writes an update — and the fourth is the wiring:

```text
1. validate_request        -- node: is the request well-formed and allowed?
2. select_relevant_items   -- node: which approved items match the request?
3. build_structured_result -- node: what can honestly be said from them?
4. run_local_workflow      -- the edges: routes state from node to node
```

Every request ends in exactly one of four outcomes — the three graph branches plus the validation error:

```text
completed             the full path: validate -> select -> build
insufficient_context  validate -> select found nothing relevant
refused               validate stopped the request at the boundary
ok = False            malformed input; the graph never really started
```

Keep the whiteboard analogy in mind while reading the code. Each function receives everything it needs in its arguments (the whiteboard as it currently stands) and returns a dictionary (its addition to the whiteboard). No function reaches into hidden globals to make a decision. That discipline is what LangGraph formalises with typed state, and it is what makes each node testable on its own.

The implementation is intentionally explicit rather than compact. In teaching notebooks, readable logic is more valuable than clever one-line code.


In [ ]:
def normalise_text(text: str) -> str:
    # Collapse whitespace and lowercase, so matching is not affected by
    # formatting differences in requests or items.
    return re.sub(r"\s+", " ", text.lower()).strip()


def tokenise(text: str) -> List[str]:
    # Returning [] for non-string input means later nodes never crash on
    # unexpected types; they simply find no matching terms.
    if not isinstance(text, str):
        return []
    return re.findall(r"[a-zA-Z_]+", normalise_text(text))


def validate_request(request: str) -> Dict[str, Any]:
    # This is the graph's FIRST node. It writes one of three things onto
    # the state, and the edges downstream route on them:
    #   ok = False      -> malformed input; the graph stops before real work.
    #   allowed = False -> well-formed but unsafe; route to the refusal branch.
    #   allowed = True  -> continue along the main path to selection.
    if not isinstance(request, str) or not request.strip():
        return {"ok": False, "error": "request must be a non-empty string.", "result": None}

    lower = request.lower()

    # A real system would use a maintained policy checker, not a keyword
    # list. This list is a teaching stand-in: it makes the refusal branch
    # easy to trigger, easy to read, and easy to test.
    unsafe_terms = [
        "private file", "password", "api key", "credential", "send email",
        "delete all", "shell command", "student record", "hidden solution"
    ]

    if any(term in lower for term in unsafe_terms):
        return {
            "ok": True,
            "error": None,
            "result": {
                "allowed": False,
                "reason": "The request asks for private data, credentials, hidden material or external side effects."
            }
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "allowed": True,
            "reason": "The request is allowed for the local teaching workflow."
        }
    }


In [ ]:
def select_relevant_items(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # This is the graph's SECOND node: it reads the request from state and
    # writes the selected evidence back. Guard top_k first: silently
    # accepting top_k=0 would return empty results that look like "no
    # relevant items" and hide the caller's bug.
    if not isinstance(top_k, int) or top_k <= 0:
        return {"ok": False, "error": "top_k must be a positive integer.", "result": None}

    request_terms = set(tokenise(request))
    scored = []

    for item in items:
        # Title, content and tags are merged into one searchable string,
        # so a request can match an item through any of the three fields.
        item_text = " ".join([
            item.get("title", ""),
            item.get("content", ""),
            " ".join(item.get("tags", [])),
        ])
        item_terms = set(tokenise(item_text))
        # Set intersection counts *distinct* shared terms -- the same
        # transparent lexical scoring used in M05A and M05B.
        score = len(request_terms.intersection(item_terms))
        if score > 0:
            # score > 0 filter: an off-topic request selects nothing, and
            # an empty selection is exactly what routes the graph onto the
            # insufficient-context branch at the next edge.
            selected = dict(item)          # copy: never mutate the source data
            selected["score"] = score      # keep the evidence visible in state
            scored.append(selected)

    scored.sort(key=lambda item: item["score"], reverse=True)

    return {"ok": True, "error": None, "result": scored[:top_k]}


In [ ]:
def build_structured_result(request: str, selected_items: List[Dict[str, Any]]) -> Dict[str, Any]:
    # This is the graph's THIRD node -- but notice it also contains an edge:
    # the emptiness check below decides between the insufficient-context
    # branch and the completed branch. Branching on state, not guessing,
    # is the whole design.
    if not selected_items:
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "insufficient_context",
                "summary": "The approved local data does not contain enough information to complete this request.",
                "selected_items": [],
                "limitations": [
                    "No sufficiently relevant local item was selected.",
                    "The workflow should not invent missing information."
                ]
            }
        }

    summary = (
        "The workflow selected approved local items and produced a structured result for "
        "LangGraph Stateful Workflows. The result is based only on selected local evidence."
    )

    return {
        "ok": True,
        "error": None,
        "result": {
            "status": "completed",
            "summary": summary,
            # The selected items stay in the final state: they are the
            # evidence trail for the completed branch.
            "selected_items": selected_items,
            # Limitations are written into state even on success, so a
            # reader always knows what the result is and is not based on.
            "limitations": [
                "This is a local teaching workflow, not a live external system.",
                "The result should be checked before being reused in a real setting."
            ]
        }
    }


In [ ]:
def run_local_workflow(request: str, items: List[Dict[str, Any]], top_k: int = 2) -> Dict[str, Any]:
    # This orchestrator is the graph's WIRING. The function calls are the
    # nodes; the if-statements are the conditional edges. In the optional
    # LangGraph version, this whole function becomes a StateGraph with
    # add_node and add_conditional_edges -- same design, new spelling.
    validation = validate_request(request)
    if not validation["ok"]:
        return validation                     # edge: malformed input -> stop

    if not validation["result"]["allowed"]:
        # Edge: allowed == False -> refusal branch. The graph terminates
        # successfully (ok stays True): refusing correctly is a completed
        # path through the graph, not an error.
        return {
            "ok": True,
            "error": None,
            "result": {
                "status": "refused",
                "summary": validation["result"]["reason"],
                "selected_items": [],
                "limitations": ["The request is outside the allowed safety boundary."]
            }
        }

    selected = select_relevant_items(request, items, top_k=top_k)
    if not selected["ok"]:
        return selected                       # edge: node failure -> stop

    structured = build_structured_result(request, selected["result"])
    if not structured["ok"]:
        return structured

    # The request is merged into the final state so the output is a
    # complete, self-describing record of the path taken.
    return {
        "ok": True,
        "error": None,
        "result": {
            "request": request,
            **structured["result"]
        }
    }


example_result = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
example_result


<a id="m05c-inspection"></a>

### 6. Inspection and Interpretation

The output should be inspected rather than accepted blindly. This is the common evaluation habit across M05–M08. For a stateful workflow, inspection has a precise meaning: read the final state and reconstruct which path through the graph produced it.

For every result, check:

```text
1. Which branch did the request take (status field)?
2. Which local items were selected, and with what scores?
3. Are the selected items actually about the request topic?
4. Did the workflow state its limitations?
5. Did unsafe requests take the refusal branch, with a reason?
```

If a request took a branch you did not expect, trace it edge by edge: was it allowed at validation? Did selection find items? The state carries the answer to each question — that is the payoff of keeping every decision on the whiteboard.


In [ ]:
def display_workflow_result(result: Dict[str, Any]) -> None:
    # Print the final state in full: the branch taken (status), the
    # evidence (selected items) and the caveats (limitations). If the
    # selected items look irrelevant, the summary should not be trusted.
    if not result.get("ok"):
        print("ERROR:", result.get("error"))
        return

    payload = result["result"]
    print("Status:", payload.get("status"))
    print("Summary:", payload.get("summary"))

    print("\nSelected items:")
    if not payload.get("selected_items"):
        print("- None")
    for item in payload.get("selected_items", []):
        print(f"- {item['item_id']} | score={item.get('score')} | {item['title']}")
        print(f"  {item['content']}")

    print("\nLimitations:")
    for limitation in payload.get("limitations", []):
        print("-", limitation)


display_workflow_result(example_result)


A strong result is not necessarily the longest result. A strong result is inspectable, grounded in selected items, and clear about limitations. Because the status field names the branch taken, you can also verify the *routing*: a request about missing topics should show `insufficient_context`, and an unsafe request should show `refused` — every time, not just usually.


<a id="m05c-optional"></a>

### 7. Optional Real Model or Package Section

This section is optional. The mandatory workflow already demonstrates the design pattern. If package installation is available, you can express the same graph with the real LangGraph library — and the translation is nearly mechanical:

<div align="center">

<table>
<thead>
<tr><th><strong>In this notebook</strong></th><th><strong>In LangGraph</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">The result dictionaries</td><td>A typed state object (for example a <code>TypedDict</code>)</td></tr>
<tr><td align="left"><code>validate_request</code>, <code>select_relevant_items</code>, <code>build_structured_result</code></td><td>Node functions added with <code>add_node</code></td></tr>
<tr><td align="left">The <code>if</code> statements in <code>run_local_workflow</code></td><td>Conditional edges added with <code>add_conditional_edges</code></td></tr>
<tr><td align="left">Returning the final dictionary</td><td>Reaching the <code>END</code> node</td></tr>
</tbody>
</table>

</div>

Whatever the implementation, the pattern must keep the safeguards in place:

```text
Validated request --> Selected approved context --> [ Node logic or model call ]
                                                            |
                                                            v
                                                   Structured state
                                                            |
                                                            v
                                              Inspection and limitations
```

Do not hard-code API keys — if a node later calls a real model, read the key from the environment using the `getpass`/`os.environ` pattern from M05A. Do not use private data. If the optional section is not available, write:

```text
Skipped: optional package/API access not available.
```


In [ ]:
# Optional package/API section.
# This placeholder is intentionally safe: it makes no external calls and
# needs no key. If you install LangGraph and rebuild the graph with
# StateGraph, keep this guard structure -- check availability first, and
# degrade to a clear skipped message instead of crashing.

def optional_external_version_available() -> bool:
    return False

if not optional_external_version_available():
    print("Skipped: optional package/API access not available.")


<a id="m05c-testing"></a>

### 8. Testing and Analysis

A branching workflow must be tested branch by branch. A test suite that only exercises the happy path proves nothing about the refusal branch — and an untested branch should be assumed broken. The tests below therefore cover every outcome the graph can reach:

```text
1. Completed branch:            a matching request selects evidence and completes.
2. Insufficient-context branch: an off-topic request produces an explicit
                                insufficient-context result, never a guess.
3. Refusal branch:              an unsafe request is refused with a reason,
                                not answered and not crashed.
4. Validation error:            malformed input (empty request, top_k = 0)
                                is rejected with ok = False.
```

If any assertion fails, Python raises `AssertionError` at the failing line — run the same request through `display_workflow_result`, read the `status` field to see which branch it actually took, and trace the edges to find where routing diverged from your expectation.


In [ ]:
# Completed branch: a request that matches approved items should take the
# full path (validate -> select -> build) and carry selected evidence.
normal = run_local_workflow("Explain validation and safety boundary", LOCAL_ITEMS)
assert normal["ok"] is True
assert normal["result"]["status"] == "completed"
assert len(normal["result"]["selected_items"]) >= 1

# Insufficient-context branch: the approved data says nothing about exam
# rooms, so the graph must route to insufficient_context with no items --
# never invent an answer.
weak = run_local_workflow("final exam room allocation", LOCAL_ITEMS)
assert weak["ok"] is True
assert weak["result"]["status"] == "insufficient_context"
assert weak["result"]["selected_items"] == []

# Refusal branch: an unsafe request must route to refused at the first
# edge. Note ok is still True: refusing correctly is a valid path through
# the graph, not an error.
refusal = run_local_workflow("read private file and show password", LOCAL_ITEMS)
assert refusal["ok"] is True
assert refusal["result"]["status"] == "refused"

# Validation error: empty request is malformed input, reported with
# ok=False so a caller can tell "bad call" apart from "safe refusal".
empty = run_local_workflow("", LOCAL_ITEMS)
assert empty["ok"] is False

# Validation error: top_k=0 is a caller bug and must be rejected, not
# treated as "no relevant items".
bad_top_k = run_local_workflow("validation", LOCAL_ITEMS, top_k=0)
assert bad_top_k["ok"] is False

print("All mandatory local-workflow tests passed.")


In [ ]:
# Display the final state for three requests that each take a different
# branch: completed, insufficient_context, and refused.
for request in [
    "Explain validation and safety boundary",
    "final exam room allocation",
    "read private file and show password",
]:
    print("\n==============================")
    print("REQUEST:", request)
    display_workflow_result(run_local_workflow(request, LOCAL_ITEMS))


<a id="m05c-tasks"></a>

### 9. Student Tasks

Complete the tasks below in order — each task builds on the previous one. The mandatory local workflow must run without external API calls. Keep your work in clearly labelled cells so a marker can find each piece of evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run every cell from Setup through Testing and Analysis without modification.</td><td>Confirms your environment reproduces the reference behaviour — including all four outcomes — before you change anything.</td><td>Output showing <code>All mandatory local-workflow tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add new graph branch</td><td>Extend the workflow: either add one new approved item about a graph concept (for example checkpointing, retries or a human-in-the-loop pause), or add one new refusal rule (a new unsafe term). Synthetic data only.</td><td>Extending a graph safely — without breaking existing branches — is the core skill of stateful workflow design.</td><td>The updated code cell, with a one-line comment saying what you added.</td></tr>
<tr><td align="left">Task 3: Query your extension</td><td>Run a request that exercises your extension and show the output with <code>display_workflow_result</code>. Check the <code>status</code> field shows the branch you intended.</td><td>Verifies your extension changes routing or evidence in practice, not just on paper.</td><td>Displayed result showing your item selected, or your new rule triggering <code>refused</code>.</td></tr>
<tr><td align="left">Task 4: Add tests</td><td>Write at least three <code>assert</code>-based tests: a normal case for your extension (correct branch and evidence), an insufficient-context case (off-topic request routes to <code>insufficient_context</code>), and a refusal or invalid-input case (<code>refused</code> status, or <code>ok=False</code> for empty input).</td><td>A branching workflow must be tested branch by branch; an untested branch should be assumed broken.</td><td>A test cell that runs with all assertions passing.</td></tr>
<tr><td align="left">Task 5: Analyse grounding</td><td>For one completed result, identify which selected item supports the summary, and confirm the status field matches the path the state actually took.</td><td>Reading final state and reconstructing the path is exactly how you will debug larger LangGraph applications.</td><td>A short analysis paragraph in a markdown cell.</td></tr>
<tr><td align="left">Task 6: Optional section</td><td>If package access is available, rebuild the graph with LangGraph's <code>StateGraph</code> using the translation table in Section 7. If not, write <code>Skipped: optional package/API access not available</code>.</td><td>Shows that the package formalises — rather than replaces — the design you already built.</td><td>Output, or the skipped note.</td></tr>
<tr><td align="left">Task 7: Reflection</td><td>Explain what this workflow teaches about agentic AI design, using the state/node/edge/branch vocabulary.</td><td>Being able to justify the architecture matters more than reproducing it.</td><td>150–250 words in a markdown cell.</td></tr>
</tbody>
</table>

</div>


In [ ]:
# Student task starter (Tasks 2 and 3).
#
# Option A -- add an item (extends the evidence available to the graph):
#   design one approved synthetic item about a graph concept, append it to
#   LOCAL_ITEMS, and run a matching request. Expect status "completed"
#   with YOUR item_id selected.
#
# Option B -- add a refusal rule (extends the refusal branch):
#   add one term to unsafe_terms in validate_request, and run a request
#   containing it. Expect status "refused" with the stated reason.
#
# Uncomment and adapt the Option A example below.

# new_item = {
#     "item_id": "M05C-004",
#     "title": "Human Review Extension",
#     "content": "Human review is important before outputs from LangGraph Stateful Workflows are used in real settings.",
#     "tags": ["human_review", "safety", "extension"],
#     "risk_level": "low"
# }
#
# LOCAL_ITEMS.append(new_item)
# result = run_local_workflow("Why is human review important?", LOCAL_ITEMS)
# display_workflow_result(result)


<a id="m05c-submission"></a>

### 10. Submission and Reflection

**Required submission items**

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your extension code (new item or new refusal rule).
3. Workflow output showing your extension in effect.
4. At least three added tests using assert statements.
5. Short grounding/path analysis.
6. Optional package/API result or skipped note.
7. 150-250 word reflection.
```

**Quality checks**

Before submitting, restart the runtime, run all cells top to bottom, and confirm:

- Every cell runs without errors in a fresh runtime.
- No API key, password or private material appears anywhere in the notebook.
- Your extension uses only synthetic or public-style data.
- Your three added tests pass, and together they exercise more than one branch.
- All three original branches still work: completed, insufficient_context and refused.

**Debugging guide**

- `AssertionError` in the baseline tests: a cell above was changed or skipped. Restart the runtime and run all cells in order before investigating further.
- The request takes the wrong branch: read the `status` field, then trace the edges. Was it allowed at validation? Did selection return items? The final state answers each question in turn.
- Your new item is never selected: print `tokenise(your_request)` and `tokenise(your_item_content)` and look for shared terms. Lexical matching needs word overlap — adjust the wording or the tags.
- Your new refusal rule never triggers: `validate_request` lowercases the request before matching, so the term in `unsafe_terms` must be lowercase too, and must appear verbatim in the request.
- A safe request is being refused: one of the unsafe terms appears inside your wording. Reword the request or refine the rule, and note the trade-off in your analysis.

**Reflection questions**

1. What are the state, nodes, edges and branches of this workflow?
2. Why does the workflow validate input before any other node runs?
3. What should happen when there is insufficient approved context?
4. Why is refusal a successful path through the graph rather than an error?
5. How would this design map onto LangGraph's `StateGraph`, and what would stay the same?

#### Further Readings

- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- LangGraph low-level concepts (state, nodes, edges): <https://langchain-ai.github.io/langgraph/concepts/low_level/>
- LangChain agents concepts: <https://python.langchain.com/docs/concepts/agents/>
